# Hybrid KIE — final single-model training

Notebook Kaggle cuối cùng để train lại **một** model Hybrid trên toàn bộ CORD. Kiến trúc, pipeline dữ liệu, loss, optimizer và effective batch size được giữ nguyên từ `Hybrid.ipynb`; notebook chỉ giữ quy trình huấn luyện Hybrid cuối cùng.

Thiết lập: seed 42, chạy đủ 14 epoch, batch size 2 và gradient accumulation 2. Notebook lưu đúng một checkpoint deploy, processor, labels, metrics và một file ZIP trong `/kaggle/working`.

## 1. Kaggle Runtime Setup and Environment Validation

This section installs only missing dependencies, imports the required machine-learning and graph-processing libraries, verifies CUDA availability, locates the CORD dataset, and initializes the experiment workspace. The existing Kaggle PyTorch installation is retained to maintain CUDA compatibility.


In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED = {
    "torch_geometric": "torch-geometric>=2.4,<3",
    "torchcrf": "pytorch-crf>=0.7.2,<1",
    "transformers": "transformers>=4.30,<6",
    "sklearn": "scikit-learn>=1.2,<2",
}

for import_name, package in REQUIRED.items():
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("Dependencies are ready. Restart the kernel only if Kaggle requests it.")

In [ ]:
import gc
import hashlib
import json
import math
import os
import platform
import random
import shutil
import time
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric
import transformers
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_recall_fscore_support
from scipy.stats import t as student_t
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torch_geometric.data import Batch, Data
from torch_geometric.nn import GATv2Conv
from torchcrf import CRF
from tqdm.auto import tqdm
from transformers import LayoutLMv3Model, LayoutLMv3Processor, get_linear_schedule_with_warmup

assert torch.cuda.is_available(), "Hãy bật GPU trong Kaggle trước khi train."

print("Runtime: Kaggle | Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("Transformers:", transformers.__version__)
print("PyG:", torch_geometric.__version__)
print("scikit-learn:", sklearn.__version__)


## 2. Experimental Configuration and Evaluation Protocol

This section defines the architecture, optimization schedule, regularization, graph parameters, reproducibility controls, and artifact paths.

The model is trained exactly five times using seeds `13`, `42`, `2026`, `7`, and `123`. Every run uses the same fixed configuration: two Relation-GATv2 layers, graph scale `0.50`, base auxiliary weight `0.35`, and early-stopping patience `6`. No hyperparameter sweep is performed within this experiment.

Each seed starts from the pretrained `microsoft/layoutlmv3-base` checkpoint with independently initialized graph, fusion, classification, and CRF parameters. Development Entity Macro F1 in the original graph mode is the sole checkpoint-selection criterion.


In [ ]:
@dataclass(frozen=True)
class Config:
    profile: str = "final_single_model"
    model_id: str = "microsoft/layoutlmv3-base"
    model_revision: str = "main"
    max_length: int = 512
    batch_size: int = 2
    grad_accum_steps: int = 2
    num_workers: int = 2

    backbone_lr: float = 1e-5
    head_lr: float = 1e-4
    graph_lr: float = 2e-4
    weight_decay: float = 0.01
    warmup_ratio: float = 0.10
    max_epochs: int = 14
    patience: int = 14
    min_delta: float = 1e-4
    freeze_backbone_epochs: int = 1
    lm_dropout: float = 0.30
    gat_attention_dropout: float = 0.10
    graph_dropout: float = 0.10
    fusion_dropout: float = 0.10
    gat_layers: int = 2
    gat_heads: int = 4
    gat_edge_hidden: int = 64
    gat_ffn_multiplier: int = 1
    gat_residual_scale: float = 0.50
    spatial_input_scale: float = 0.10
    relation_types: int = 7
    edge_dim: int = 18
    edge_dropout: float = 0.00

    fixed_graph_alpha: float = 0.50
    base_aux_weight: float = 0.35
    graph_branch_dropout: float = 0.10
    max_graph_neighbors: int = 8
    column_center_threshold: float = 0.10
    column_overlap_threshold: float = 0.20

    crf_weight: float = 0.65
    focal_gamma: float = 2.0
    knn_k: int = 2
    use_rare_document_sampler: bool = True
    rare_sampler_power: float = 0.50
    rare_sampler_max_weight: float = 2.50
    class_balance_beta: float = 0.999
    class_weight_cap: float = 3.0
    use_amp: bool = True
    keep_all_checkpoints: bool = True
    artifact_dir: str = "/kaggle/working/receipt_kie_hybrid_final"


CFG = Config()
SEEDS = [42]
MAX_EPOCHS = CFG.max_epochs
PATIENCE = CFG.patience
SELECTED_ALPHA = CFG.fixed_graph_alpha
SELECTED_BASE_AUX = CFG.base_aux_weight
MODEL_NAME = "layoutlmv3_split_head_symbolic_rel_gatv2_fusion_crf"
DEVICE = torch.device("cuda")
ARTIFACT_DIR = Path(CFG.artifact_dir)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(
    "Final single-model protocol:",
    "| seeds:", SEEDS,
    "| graph alpha:", SELECTED_ALPHA,
    "| base auxiliary weight:", SELECTED_BASE_AUX,
    "| patience:", PATIENCE,
    "| train runs:", len(SEEDS),
    "| initialization: pretrained LayoutLMv3 + random heads/graph",
)


In [ ]:
def set_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def find_cord_root() -> Path:
    candidates = [Path("CORD1000/CORD/CORD")]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        candidates.extend(path.parent.parent for path in kaggle_input.rglob("train/json"))
    for root in candidates:
        if all(
            (root / split / "json").is_dir() and (root / split / "image").is_dir()
            for split in ("train", "dev", "test")
        ):
            return root.resolve()
    raise FileNotFoundError(
        "Không tìm thấy CORD root có train/dev/test/{image,json}. "
        "Hãy attach dataset CORD-1000 vào Kaggle Notebook."
    )


set_seed(SEEDS[0])
CORD_ROOT = find_cord_root()
print("CORD root:", CORD_ROOT)

## 3. Label Schema and Dataset Audit

This section maps CORD annotations to 18 receipt entity classes and the background class `O`. It validates image–annotation pairing for every split and summarizes document counts, word counts, and class frequencies.

Entity Macro F1 over the 18 non-background classes is the primary metric. Entity Micro F1, weighted F1, precision, recall, per-class support, and graph-specific diagnostics are retained for complementary analysis.


In [ ]:
CORD_CATEGORY_TO_LABEL = {
    "menu.nm": "S-MENU_NM", "menu.sub_nm": "S-MENU_NM",
    "menu.cnt": "S-MENU_CNT", "menu.sub_cnt": "S-MENU_CNT",
    "menu.num": "S-MENU_NUM", "menu.unitprice": "S-MENU_UNITPRICE",
    "menu.price": "S-MENU_PRICE", "menu.sub_price": "S-MENU_PRICE",
    "menu.discountprice": "S-MENU_DISCOUNT_PRICE",
    "sub_total.subtotal_price": "S-SUBTOTAL",
    "sub_total.discount_price": "S-DISCOUNT",
    "sub_total.tax_price": "S-TAX", "sub_total.service_price": "S-SERVICE",
    "total.total_price": "S-TOTAL", "total.cashprice": "S-CASH",
    "total.changeprice": "S-CHANGE", "total.creditcardprice": "S-CARD_PAYMENT",
    "total.emoneyprice": "S-EMONEY_PAYMENT", "total.menuqty_cnt": "S-MENUQTY_CNT",
    "total.menutype_cnt": "S-MENUTYPE_CNT",
    "sub_total.etc": "S-OTHER", "total.total_etc": "S-OTHER",
    "menu.etc": "S-OTHER", "menu.sub_etc": "S-OTHER",
    "menu.itemsubtotal": "O", "menu.vatyn": "O",
    "sub_total.othersvc_price": "O", "void_menu.nm": "O",
    "void_menu.price": "O", "menu.sub_unitprice": "O",
}

LABELS = [
    "O", "S-MENU_NM", "S-MENU_CNT", "S-MENU_NUM", "S-MENU_UNITPRICE",
    "S-MENU_PRICE", "S-MENU_DISCOUNT_PRICE", "S-SUBTOTAL", "S-DISCOUNT",
    "S-TAX", "S-SERVICE", "S-TOTAL", "S-CASH", "S-CHANGE",
    "S-CARD_PAYMENT", "S-EMONEY_PAYMENT", "S-MENUQTY_CNT",
    "S-MENUTYPE_CNT", "S-OTHER",
]
ID2LABEL = dict(enumerate(LABELS))
LABEL2ID = {label: idx for idx, label in ID2LABEL.items()}
ENTITY_IDS = list(range(1, len(LABELS)))
NUM_LABELS = len(LABELS)


def dataset_audit(root: Path) -> pd.DataFrame:
    rows = []
    for split in ("train", "dev", "test"):
        json_files = sorted((root / split / "json").glob("*.json"))
        image_files = sorted((root / split / "image").glob("*.png"))
        json_stems, image_stems = {p.stem for p in json_files}, {p.stem for p in image_files}
        counts = Counter()
        for path in json_files:
            data = json.loads(path.read_text(encoding="utf-8"))
            for line in data.get("valid_line", []):
                mapped = CORD_CATEGORY_TO_LABEL.get(line.get("category", ""), "O")
                counts[mapped] += len(line.get("words", []))
        rows.append({
            "split": split, "json": len(json_files), "images": len(image_files),
            "missing_images": len(json_stems - image_stems),
            "missing_json": len(image_stems - json_stems),
            "words": sum(counts.values()), **counts,
        })
    return pd.DataFrame(rows).fillna(0)


audit_df = dataset_audit(CORD_ROOT)
display(audit_df)
assert (audit_df[["missing_images", "missing_json"]].to_numpy() == 0).all(), "Dataset thiếu cặp image/json."


## 4. Symbolic Graph Construction and Word-Level Inputs

This section builds one graph for each receipt and prepares the corresponding LayoutLMv3 inputs.

Each graph node represents a CORD word. The graph uses seven relation types:

- `SAME_LINE` connects words that share the same `valid_line` identifier.
- `NEXT_LINE_COLUMN` connects aligned words across consecutive lines using horizontal overlap and center-distance constraints.
- `LEFT`, `RIGHT`, `ABOVE`, and `BELOW` encode directional spatial relationships.
- `KNN` supplies local fallback edges when symbolic and directional relations do not fill the neighbor budget.

Every edge also carries continuous geometric attributes derived from relative position, distance, size, overlap, and orientation. Graph construction uses document structure and geometry only; entity labels are never used to create edges.

CORD text and normalized bounding boxes are encoded with `apply_ocr=False`. LayoutLMv3 subtokens are mapped back to words, truncated words are removed consistently from labels and graphs, and variable-size graphs are batched with PyTorch Geometric.


In [ ]:
def quad_to_bbox(quad, width, height):
    xs = [quad[f"x{i}"] for i in range(1, 5)]
    ys = [quad[f"y{i}"] for i in range(1, 5)]
    x0, y0, x1, y1 = int(min(xs)), int(min(ys)), int(max(xs)), int(max(ys))
    x0, y0 = max(0, min(x0, width - 1)), max(0, min(y0, height - 1))
    x1, y1 = max(x0 + 1, min(x1, width)), max(y0 + 1, min(y1, height))
    return x0, y0, x1, y1


def build_records(root: Path, split: str):
    records = []
    for json_path in tqdm(sorted((root / split / "json").glob("*.json")), desc=f"Parse {split}"):
        data = json.loads(json_path.read_text(encoding="utf-8"))
        width = int(data["meta"]["image_size"]["width"])
        height = int(data["meta"]["image_size"]["height"])
        image_path = root / split / "image" / f"{json_path.stem}.png"
        words = []
        for line_index, line in enumerate(data.get("valid_line", [])):
            label = CORD_CATEGORY_TO_LABEL.get(line.get("category", ""), "O")
            for word in line.get("words", []):
                text = word.get("text", "").strip()
                if text:
                    raw_row_id = word.get("row_id")
                    line_id = (
                        f"row_{raw_row_id}"
                        if raw_row_id is not None
                        else f"valid_line_{line_index}"
                    )
                    words.append({
                        "text": text,
                        "box": quad_to_bbox(word["quad"], width, height),
                        "label": label,
                        "line_id": line_id,
                    })
        words.sort(key=lambda item: (item["box"][1], item["box"][0]))
        if words and image_path.exists():
            records.append({
                "id": json_path.stem,
                "image_path": image_path,
                "size": (width, height),
                "words": words,
            })
    return records


train_records = build_records(CORD_ROOT, "train")
dev_records = build_records(CORD_ROOT, "dev")
test_records = build_records(CORD_ROOT, "test")
print("Documents:", len(train_records), len(dev_records), len(test_records))

In [ ]:
from app.graph import (
    INVERSE_RELATION,
    RELATION_TO_ID,
    build_spatial_graph,
    horizontal_overlap_ratio,
    horizontally_aligned,
    normalize_box,
    vertical_overlap_ratio,
)

In [ ]:
processor = LayoutLMv3Processor.from_pretrained(
    CFG.model_id, revision=CFG.model_revision, apply_ocr=False
)


class CORDWordDataset(Dataset):
    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        width, height = record["size"]
        image = Image.open(record["image_path"]).convert("RGB")
        texts = [item["text"] for item in record["words"]]
        boxes = [normalize_box(item["box"], width, height) for item in record["words"]]
        encoding = processor(
            image, texts, boxes=boxes, truncation=True, max_length=CFG.max_length,
            padding="max_length", return_tensors="pt",
        )
        word_ids = encoding.word_ids(batch_index=0)
        active_ids = sorted({wid for wid in word_ids if wid is not None})
        ranges = []
        for wid in active_ids:
            positions = [pos for pos, current in enumerate(word_ids) if current == wid]
            ranges.append((positions[0], positions[-1] + 1))
        active_words = [record["words"][wid] for wid in active_ids]
        labels = torch.tensor([LABEL2ID[item["label"]] for item in active_words], dtype=torch.long)
        graph = build_spatial_graph(active_words, record["size"], CFG.knn_k)
        return {
            "id": record["id"],
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "bbox": encoding["bbox"].squeeze(0),
            "pixel_values": encoding["pixel_values"].squeeze(0),
            "word_ranges": ranges, "word_labels": labels, "graph": graph,
        }


def collate_fn(samples):
    tensor_keys = ("input_ids", "attention_mask", "bbox", "pixel_values")
    batch = {key: torch.stack([sample[key] for sample in samples]) for key in tensor_keys}
    max_words = max(len(sample["word_labels"]) for sample in samples)
    labels = torch.zeros((len(samples), max_words), dtype=torch.long)
    mask = torch.zeros((len(samples), max_words), dtype=torch.bool)
    for i, sample in enumerate(samples):
        size = len(sample["word_labels"])
        labels[i, :size] = sample["word_labels"]
        mask[i, :size] = True
    batch.update({
        "ids": [sample["id"] for sample in samples],
        "word_ranges": [sample["word_ranges"] for sample in samples],
        "word_labels": labels, "word_mask": mask,
        "graphs": Batch.from_data_list([sample["graph"] for sample in samples]),
    })
    return batch


train_ds, dev_ds, test_ds = map(CORDWordDataset, (train_records, dev_records, test_records))


def build_document_sample_weights(records):
    document_labels = [
        {LABEL2ID[word["label"]] for word in record["words"] if word["label"] != "O"}
        for record in records
    ]
    document_frequency = Counter(label for labels in document_labels for label in labels)
    weights = []
    for labels in document_labels:
        rarity = [
            (len(records) / max(document_frequency[label], 1)) ** CFG.rare_sampler_power
            for label in labels
        ]
        weights.append(min(max(rarity, default=1.0), CFG.rare_sampler_max_weight))
    return torch.tensor(weights, dtype=torch.double), document_frequency


train_sample_weights, train_document_frequency = build_document_sample_weights(train_records)
print("Rare-document sampler:", CFG.use_rare_document_sampler,
      "| weight range:", f"{train_sample_weights.min():.2f}–{train_sample_weights.max():.2f}")


def make_loaders(seed):
    loader_generator = torch.Generator().manual_seed(seed)
    common = dict(batch_size=CFG.batch_size, num_workers=CFG.num_workers,
                  pin_memory=True, collate_fn=collate_fn,
                  persistent_workers=CFG.num_workers > 0)
    if CFG.use_rare_document_sampler:
        sampler_generator = torch.Generator().manual_seed(seed)
        sampler = WeightedRandomSampler(
            train_sample_weights, num_samples=len(train_ds), replacement=True,
            generator=sampler_generator,
        )
        train_loader = DataLoader(train_ds, sampler=sampler, generator=loader_generator, **common)
    else:
        train_loader = DataLoader(train_ds, shuffle=True, generator=loader_generator, **common)
    return (
        train_loader,
        DataLoader(dev_ds, shuffle=False, **common),
        DataLoader(test_ds, shuffle=False, **common),
    )


## 5. Two-Layer Relation-GATv2, Split Heads, and Hidden Fusion

This section defines the hybrid architecture.

LayoutLMv3 first produces one contextual representation per word. A normalized graph input combines these representations with a small projected spatial signal. Two Relation-GATv2 blocks then perform edge-conditioned message passing. Each block contains pre-normalization, a learned edge encoder, multi-head GATv2 attention, residual connections, and a feed-forward residual sublayer.

The graph branch models a correction rather than replacing the language-layout representation. The change introduced by message passing is projected and fused with the original LayoutLMv3 word state through a fixed-scale residual with `alpha = 0.50`. The final fusion projection is zero-initialized so training begins from the non-graph representation.

Separate linear classifiers are used for the base and fused branches. This prevents the auxiliary base objective from directly shifting the decision boundary of the graph-enhanced classifier. A shared word-level CRF provides structured decoding for both paths. Attention entropy, attention variance, residual magnitude, and residual-to-base norm ratio are recorded as graph diagnostics.


In [ ]:
from app.model import (
    LayoutLMv3SymbolicRelationGATFusionCRF,
    RelationEdgeGATBlock,
    WordModelBase,
)

Checkpoint selection uses development Entity Macro F1 from the trained Hybrid model only.


In [ ]:
train_label_counts = Counter(
    LABEL2ID[word["label"]] for record in train_records for word in record["words"]
)
counts = torch.tensor([train_label_counts.get(i, 0) for i in range(NUM_LABELS)], dtype=torch.float32)
beta = CFG.class_balance_beta
effective_counts = (1 - torch.pow(torch.tensor(beta), counts.clamp_min(1))) / (1 - beta)
class_weights = effective_counts.reciprocal()
entity_mean = class_weights[1:].mean()
class_weights = class_weights / entity_mean
class_weights[0] = 1.0
class_weights = class_weights.clamp(max=CFG.class_weight_cap).to(DEVICE)
display(pd.DataFrame({"label": LABELS, "train_words": counts.int(), "weight": class_weights.cpu()}))


def calculate_metrics(golds, preds):
    precision, recall, macro_f1, _ = precision_recall_fscore_support(
        golds, preds, labels=ENTITY_IDS, average="macro", zero_division=0
    )
    return {
        "entity_macro_precision": float(precision),
        "entity_macro_recall": float(recall),
        "entity_macro_f1": float(macro_f1),
        "entity_micro_f1": float(f1_score(golds, preds, labels=ENTITY_IDS, average="micro", zero_division=0)),
        "weighted_f1": float(f1_score(golds, preds, labels=list(range(NUM_LABELS)), average="weighted", zero_division=0)),
    }


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    if hasattr(model, "reset_graph_diagnostics"):
        model.reset_graph_diagnostics()
    losses, golds, preds = [], [], []
    for batch in tqdm(loader, leave=False, desc="evaluate"):
        labels = batch["word_labels"].to(DEVICE)
        mask = batch["word_mask"].to(DEVICE)
        with torch.amp.autocast(device_type="cuda", enabled=CFG.use_amp):
            emissions = model(batch)
            loss = model.loss(emissions, labels, mask, class_weights)
        losses.append(loss.item())
        paths = model.decode(emissions, mask)
        for i, path in enumerate(paths):
            length = int(mask[i].sum())
            preds.extend(path[:length])
            golds.extend(labels[i, :length].tolist())
    metrics = calculate_metrics(golds, preds)
    metrics["loss"] = float(np.mean(losses))
    if hasattr(model, "graph_diagnostics"):
        metrics.update(model.graph_diagnostics())
    report = classification_report(
        golds, preds, labels=list(range(NUM_LABELS)), target_names=LABELS,
        zero_division=0, output_dict=True,
    )
    return metrics, report, golds, preds


def parameter_groups(model):
    groups = {"backbone": [], "base_head": [], "graph": []}
    for name, parameter in model.named_parameters():
        if name.startswith("backbone."):
            groups["backbone"].append(parameter)
        elif name.startswith(("spatial_proj.", "graph_input_norm.", "gat.", "graph_proj.", "fusion_proj.", "fusion_norm.")):
            groups["graph"].append(parameter)
        else:
            groups["base_head"].append(parameter)
    specs = [
        ("backbone", CFG.backbone_lr, CFG.weight_decay),
        ("base_head", CFG.head_lr, CFG.weight_decay),
        ("graph", CFG.graph_lr, CFG.weight_decay),
    ]
    return [
        {"params": groups[name], "lr": learning_rate, "weight_decay": weight_decay}
        for name, learning_rate, weight_decay in specs if groups[name]
    ]


def train_model(
    model, train_loader, dev_loader, run_name, seed, patience
):
    model.to(DEVICE)
    optimizer = AdamW(parameter_groups(model))
    steps_per_epoch = math.ceil(len(train_loader) / CFG.grad_accum_steps)
    total_steps = steps_per_epoch * MAX_EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer, int(total_steps * CFG.warmup_ratio), total_steps
    )
    scaler = torch.amp.GradScaler("cuda", enabled=CFG.use_amp)
    checkpoint_path = ARTIFACT_DIR / f"{run_name}_seed{seed}.pt"
    history, best_f1, best_epoch, stale_epochs = [], -1.0, -1, 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        model.set_graph_mode("original")
        backbone_trainable = epoch > CFG.freeze_backbone_epochs
        model.set_backbone_trainable(backbone_trainable)
        optimizer.zero_grad(set_to_none=True)
        epoch_losses = []
        progress = tqdm(train_loader, desc=f"{run_name} seed={seed} epoch={epoch}")
        for step, batch in enumerate(progress, start=1):
            labels = batch["word_labels"].to(DEVICE)
            mask = batch["word_mask"].to(DEVICE)
            with torch.amp.autocast(device_type="cuda", enabled=CFG.use_amp):
                emissions = model(batch)
                loss = model.loss(emissions, labels, mask, class_weights)
                scaled_loss = loss / CFG.grad_accum_steps
            scaler.scale(scaled_loss).backward()
            if step % CFG.grad_accum_steps == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scale_before = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scaler.get_scale() >= scale_before:
                    scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            epoch_losses.append(loss.item())
            progress.set_postfix(loss=f"{np.mean(epoch_losses[-20:]):.4f}")

        model.set_graph_mode("original")
        dev_metrics, _, _, _ = evaluate(model, dev_loader)
        row = {
            "epoch": epoch,
            "backbone_trainable": backbone_trainable,
            "train_loss": float(np.mean(epoch_losses)),
            **{f"dev_{key}": value for key, value in dev_metrics.items()},
        }
        history.append(row)
        print(row)

        current_f1 = dev_metrics["entity_macro_f1"]
        if current_f1 > best_f1 + CFG.min_delta:
            best_f1, best_epoch, stale_epochs = current_f1, epoch, 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            stale_epochs += 1
            if stale_epochs >= patience:
                print(f"Early stopping at epoch {epoch}.")
                break

    state = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    model.load_state_dict(state, strict=True)
    model.to(DEVICE).eval()
    pd.DataFrame(history).to_csv(
        ARTIFACT_DIR / f"{run_name}_seed{seed}_history.csv", index=False
    )
    return model, checkpoint_path, history, best_epoch


## Train một model và đánh giá

Cell dưới đây train đúng 14 epoch, dùng Dev Entity Macro F1 để chọn checkpoint tốt nhất, sau đó đánh giá dev/test và lưu gói deploy tối thiểu.

In [ ]:
FAST_SEED = SEEDS[0]
RUN_NAME = "hybrid_final"

set_seed(FAST_SEED)
train_loader, dev_loader, test_loader = make_loaders(FAST_SEED)
torch.cuda.reset_peak_memory_stats()
started = time.time()

model = LayoutLMv3SymbolicRelationGATFusionCRF(
    graph_alpha=SELECTED_ALPHA,
    base_aux_weight=SELECTED_BASE_AUX,
)
model, deploy_checkpoint, history, best_epoch = train_model(
    model=model,
    train_loader=train_loader,
    dev_loader=dev_loader,
    run_name=RUN_NAME,
    seed=FAST_SEED,
    patience=PATIENCE,
)
assert len(history) == MAX_EPOCHS == 14, "Training did not complete all 14 epochs"

model.set_graph_mode("original")
dev_metrics, dev_report, _, _ = evaluate(model, dev_loader)
test_metrics, test_report, _, _ = evaluate(model, test_loader)
elapsed_minutes = (time.time() - started) / 60

summary = {
    "seed": FAST_SEED,
    "best_epoch": best_epoch,
    "epochs_ran": len(history),
    "elapsed_minutes": elapsed_minutes,
    "checkpoint": str(deploy_checkpoint),
    "dev_entity_macro_f1": dev_metrics["entity_macro_f1"],
    "test_entity_macro_f1": test_metrics["entity_macro_f1"],
    "test_entity_micro_f1": test_metrics["entity_micro_f1"],
}
(ARTIFACT_DIR / "metrics.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
)
(ARTIFACT_DIR / "classification_reports.json").write_text(
    json.dumps({"dev": dev_report, "test": test_report}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
(ARTIFACT_DIR / "labels.json").write_text(
    json.dumps({"labels": LABELS, "label2id": LABEL2ID}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
processor.save_pretrained(ARTIFACT_DIR / "processor")

print("Training complete")
display(pd.Series(summary))

## Đóng gói để tải từ Kaggle

ZIP chỉ chứa checkpoint tốt nhất, history, processor, labels và metrics. Sau khi chạy, dùng **Save Version**, rồi tải file trong tab **Output** của Kaggle.

In [ ]:
metadata = {
    "task": "CORD receipt key information extraction",
    "architecture": MODEL_NAME,
    "model_id": CFG.model_id,
    "model_revision": CFG.model_revision,
    "seed": FAST_SEED,
    "graph_alpha": SELECTED_ALPHA,
    "base_aux_weight": SELECTED_BASE_AUX,
    "checkpoint": deploy_checkpoint.name,
    "input_requirement": "receipt image plus OCR words and bounding boxes",
}
(ARTIFACT_DIR / "run_metadata.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8"
)

archive_base = Path("/kaggle/working/hybrid_final_model")
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=ARTIFACT_DIR)
print("ZIP ready:", archive_path)
print("Size (MiB):", round(Path(archive_path).stat().st_size / 1024**2, 1))

### Kết quả đầu ra

- Model: `/kaggle/working/receipt_kie_hybrid_final/hybrid_final_seed42.pt`
- File tải: `/kaggle/working/hybrid_final_model.zip`

Để tránh mất file khi Kaggle kết thúc session, chọn **Save Version** sau khi cell cuối chạy xong.